# Exercise 0: How Does an LLM API Call Work?

This notebook shows the basics of how we talk to a large language model and how "tool calling" actually works under the hood. By the end you should see that there is no magic: an LLM only generates text, and everything else (memory, tools, structured fields) is code that wraps around it.

What you will work through:

1. **A Single LLM Call**: the ollama wrapper, chat templates, and how `thinking` and `content` are parsed out of raw text.
2. **Statelessness**: the API has no memory; a "conversation" is just the history you resend on each call.
3. **Tool Calling**: tools are not a special model feature, just structured text output the model was trained to produce.
4. **The Tool-Use Loop**: the cycle an agent automates (call, check, execute, send result back, call again).

Prerequisite: complete the README setup.

Verify ollama is running and the model is loaded. If this fails, nothing below will work.

In [ ]:
import ollama

models = [m.model for m in ollama.list().models]
assert any("qwen3.5" in m for m in models), f"qwen3.5:4b not found: {models}"
print("OK")

## Part 1: A Single LLM Call

This part is about the **ollama wrapper** we use to call the model. To see what ollama actually does for you, and why a wrapper is needed at all, we will look inside `ollama.chat()` step by step.

An LLM only generates raw text. The convenient fields you see (`thinking`, `content`, `tool_calls`) are parsed out of that text after generation. Inside `ollama.chat()` three layers do this work:

1. **Render**: serializes your `messages` into a single prompt string using the model's chat template.
2. **Complete**: the model produces raw text, token by token, until it emits an end-of-turn token.
3. **Parse**: splits the raw output on `<think>...</think>` into `thinking` and `content`. Tool calls (Part 3) work the same way.

In the cells below we call the lower-level `/api/generate` endpoint, so you can see the raw text directly, before parsing.

<details><summary>Where this lives in the ollama source</summary>

`renderPrompt` → `r.Completion` → `Qwen35Parser.Add` inside [ollama's `ChatHandler`](https://github.com/ollama/ollama/blob/main/server/routes.go).

</details>

Raw output via `/api/generate`. The prompt below uses Qwen3.5's chat template:

- `<|im_start|>user\n...<|im_end|>` for a user turn
- `<|im_start|>assistant\n...<|im_end|>` for the model's reply
- `<|im_start|>system\n...<|im_end|>` for system instructions
- `<think>...</think>` for reasoning, separate from the answer

Prefilling `<think>\n` makes the model start inside the thinking block.

**Goal:** see the unparsed text. `thinking` and `content` are produced by splitting this stream, not returned as separate fields.

<details><summary>What are these tokens, and why are they model-specific?</summary>

The strings above (`<|im_start|>`, `<|im_end|>`, `<think>`) are **special tokens**: short markers the model was trained to recognize as conversation structure. You normally never see them, since `ollama.chat()` and cloud APIs hide them behind a clean `{role, content}` interface.

They are **model-specific**. Qwen3.5 uses the ChatML format. Llama, Mistral, and other model families have their own conventions, baked in at training time. A template written for one model will not work for another, and using the wrong template silently degrades output quality.

</details>

<details><summary>Why prompt injection works</summary>

These tokens are just text the model was trained to recognize. If user-supplied content contains a fake `<|im_end|>` or `<|im_start|>system`, the model cannot distinguish it from the trusted template tokens. Production code usually sanitizes user input to prevent exactly this.

</details>

In [ ]:
# Raw LLM output directly via the REST API
# raw=true: no parsing by Ollama, we see the text as it is generated
# Prompt in the Qwen3.5 chat template with <think> pre-fill (activates think mode)
import requests, json

OLLAMA_URL = str(ollama.Client()._client.base_url).rstrip("/")

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "<|im_start|>user\nWhat is 2 + 2?<|im_end|>\n<|im_start|>assistant\n<think>\n",
    "raw": True,
    "stream": False,
})

# <think> was pre-filled, so prepend it for the complete picture
raw_output = "<think>\n" + resp.json()["response"]
# print(repr(raw_output))
print(raw_output)

Same question without the chat template, and the model rambles.

**Goal:** see why the template matters. Without the `<|im_start|>...<|im_end|>` structure, the model treats the prompt as a plain text completion. It isn't in "assistant turn" mode, so there's no end-of-turn signal it's trained to emit. It just keeps writing until we cut it off.

In [ ]:
# Without chat template: the model doesn't know where the answer should stop
# We abort after 3 seconds
import time

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "What is 2 + 2?",
    "raw": True,
    "stream": True,
}, stream=True)

tokens = []
start = time.time()
for line in resp.iter_lines():
    if time.time() - start > 3:
        break
    tokens.append(json.loads(line)["response"])

# print(repr("".join(tokens)))
print("".join(tokens))

`ollama.chat()` renders the template and parses the output for you.

**Goal:** see how the convenient API does all three layers automatically: your `messages` get wrapped in `<|im_start|>...<|im_end|>` tokens, the LLM produces text, and the parser splits out `thinking` and `content`.

In [ ]:
# ollama.chat() does the same thing, but:
# 1. Builds the prompt with special tokens automatically (RENDERER)
# 2. Parses <think>...</think> from the output into the "thinking" field (PARSER)
# 3. The rest becomes "content"
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "What is 2 + 2?"}],
)

print(json.dumps(response.message.model_dump(), indent=2, default=str))

---
### Your Turn

Make each change in the scratchpad below, run it, and briefly note what changed and why.

1. **Remove** the `<think>\n` from the prompt in the raw-API call (the cell with the prefilled chat template). What disappears from the output?
2. **Change** the prompt in the no-template call to `"Answer once, then stop: What is 2 + 2?"`. Does it still run away? What does this tell you about what actually stops generation?

In [ ]:
# Your Turn: Part 1
# Copy the relevant cell above, paste it here, modify, and run.


## Part 2: Statelessness

The chat API is stateless. The model only sees the `messages` we send in each call.

**Context window:** every call ships the full `messages` list (system prompt + all prior turns + tool results) through the model's fixed token budget, called the **context window** and set by `num_ctx`. Typical models support 4k to 128k tokens (a token ≈ ¾ of an English word). The model re-reads everything from scratch each call. As the chat grows, history fills the window, and eventually old messages must be dropped or summarized. This is why agents need memory strategies.

Two stateless calls. The model has no memory between them. **Goal:** confirm the API has no session state; what the model knows comes from `messages` alone.

In [ ]:
# Without conversation history: two separate calls
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "My name is Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "What is my name?"}])
print("Call 2:", r2.message.content)

Same calls, but with the previous turn passed back as history. **Goal:** see that "memory" is just history we assemble and resend.

In [ ]:
# With conversation history: same two calls, but we pass the history along
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "My name is Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[
    {"role": "user", "content": "My name is Max."},
    {"role": "assistant", "content": r1.message.content},
    {"role": "user", "content": "What is my name?"},
])
print("Call 2:", r2.message.content)

---
### Your Turn

Make each change in the scratchpad below, run it, and briefly note what changed and why.

1. **Remove** the `{"role": "assistant", ...}` entry from the message list in the with-history call's second `ollama.chat`. Does the model still know the name? Why?
2. **Replace** `r1.message.content` in the history with the hard-coded string `"Nice to meet you, Anna!"`. What does the model think your name is?
3. **Move** the fact `"My name is Max."` out of the first `user` message and into a `system` message at the start of the list. Send only `"What is my name?"` as the user turn (no assistant history). Does it still work? Which role do instructions and stable facts conventionally live in, and why is that different from facts the user states mid-conversation?


In [ ]:
# Your Turn: Part 2
# Copy the relevant cell above, paste it here, modify, and run.


## Part 3: Tool Calling

An LLM always only generates text. Tool calling is not a special feature, but a **convention for structured output**: the model was trained to output JSON instead of free text for certain prompts.

Tool calling by hand: a system prompt asks the model for JSON. **Goal:** see that tool calling is not a special model feature, just structured text output.

In [ ]:
# "Tool calling" without the tools= parameter, using only a prompt
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[
        {"role": "system", "content": 'When you need to calculate, respond ONLY with this JSON:\n{"tool": "calculator", "expression": "<expression>"}\nNo other text.'},
        {"role": "user", "content": "Calculate 17 * 23"},
    ],
)

print("=== Raw Output (just text) ===")
print(response.message.content)

Parse that JSON ourselves and run the expression. **Goal:** see that the LLM only decides what to call; execution is on us.

In [ ]:
# We can parse the text output ourselves and execute the tool
import json as _json

parsed = _json.loads(response.message.content)
print("Parsed:", parsed)

# This is tool calling: the model outputs structured text, we parse and execute.
print(parsed["expression"])
result = eval(parsed["expression"])
print("Result:", result)

This works, but it's fragile. Both this approach and the `tools=` parameter end up putting tool info into the prompt, so why is hand-rolling fragile?

- **By hand:** you invent the format. The model treats it as a generic instruction and often deviates: it adds preamble ("Here's the JSON: ..."), wraps the output in code fences, mangles keys, or refuses on edge cases. You then have to robustly parse whatever it emits.
- **With `tools=`:** ollama injects the tool info in the **exact format the model was fine-tuned on**. Qwen3.5 has seen thousands of (tool_def → tool_call) examples in this format and learned to emit a specific, reliably-parseable structure. Ollama's parser then pulls the call out into a structured `tool_calls` field.

From here on we use `tools=`. The **tool definition** is the JSON schema describing the function. The **tool implementation** is the Python function we execute. They're connected via `tool_map`.

Tool definition: the JSON schema we hand to the model. **Goal:** see what the model needs to know about an available function (name, description, parameter shape).

In [ ]:
# Tool definition: describes to the model which function is available
# Reference: https://github.com/ollama/ollama/blob/main/docs/api.md#chat-request-with-tools
calculator_tool = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression"}
                },
                "required": ["expression"],
            },
        },
    }
]

With `tools=`, the model returns `tool_calls` instead of text. **Goal:** see how the structured output now appears as a parsed field, not as raw JSON in `content`.

In [ ]:
# Call the LLM with the tool definition, inspect the full response
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Calculate 17 * 23"}],
    tools=calculator_tool,
)

print(json.dumps(response.model_dump(), indent=2, default=str))

In the response we see:
- `content` is empty: the model did not return any text
- `tool_calls` is populated: the model wants to call `calculator` with `{"expression": "17 * 23"}`

The model did not calculate anything. It returned **which function** it wants to call with **which arguments**. Now we need the Python function that actually executes it.

Tool implementation: the Python function we actually run. **Goal:** distinguish the schema (what the model sees) from the code (what we execute).

In [ ]:
# Tool implementation: the Python function we execute
import numexpr, math

def calculator(expression: str) -> str:
    result = numexpr.evaluate(expression, local_dict={"pi": math.pi, "e": math.e})
    return f"Result: {float(result)}"

# Test: call directly
print(calculator("17 * 23"))

Dispatch: look up the function by name, call it with the model's arguments. **Goal:** see how a simple name→function dict wires the model's choice to real code.

In [ ]:
# Connection: how do we find the right function for a tool call?
# The name in the tool call ("calculator") must match the key in tool_map.

tool_map = {"calculator": calculator}

tc = response.message.tool_calls[0]       # Tool call from the response
print(f"name:      {tc.function.name}")    # "calculator"
print(f"arguments: {tc.function.arguments}")  # {"expression": "17 * 23"}

func = tool_map[tc.function.name]          # Look up the calculator function
result = func(**tc.function.arguments)     # calculator(expression="17 * 23")
print(f"result:    {result}")

---
### Your Turn

Make each change in the scratchpad below, run it, and briefly note what changed and why.

1. **Change** the `description` in `calculator_tool` to an empty string `""`, then rerun the `tools=` call with prompt `"Calculate 17 * 23"`. Does the model still call the tool?
2. **Change** the user prompt in the `tools=` call to `"What is 2 + 2?"`. Does the model call the tool for trivial math, or answer directly? What's in `content` vs `tool_calls`?
3. **Rename** the tool in `calculator_tool` from `"calculator"` to `"math_thing"` but leave `tool_map = {"calculator": calculator}` unchanged in the dispatch cell. Rerun the dispatch. What error do you get, and at which line?


In [ ]:
# Your Turn: Part 3
# Copy the relevant cell above, paste it here, modify, and run.


## Part 4: Tool-Use Loop

We have a tool result, but the API is stateless: the model knows nothing about the execution. We must include the result as a `{"role": "tool"}` message in the history and call the model again.

Full loop: model requests tool → we execute → we send the result back → model answers. **Goal:** see the complete cycle an agent automates.

In [ ]:
# Complete tool-use loop
messages = [{"role": "user", "content": "Calculate 17 * 23"}]

# 1. Call the LLM
response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
print("=== Response 1 ===")
print(json.dumps(response.message.model_dump(), indent=2, default=str))

# 2. Check: does the model want to call a tool?
print("\n=== tool_calls ===")
print(response.message.tool_calls)

if response.message.tool_calls:
    # 3. Execute the tool
    tc = response.message.tool_calls[0]
    func = tool_map[tc.function.name]
    # ** unpacks the dict into keyword arguments:
    # {"expression": "17 * 23"} becomes calculator(expression="17 * 23")
    result = func(**tc.function.arguments)

    # 4. Send the tool result back to the model
    messages.append(response.message.model_dump())
    messages.append({"role": "tool", "content": result})

    print("\n=== Messages to LLM ===")
    print(json.dumps(messages, indent=2, default=str))

    response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)

print("\n=== Final Response ===")
print(json.dumps(response.model_dump(), indent=2, default=str))

Sanity check: when the model declines, `tool_calls` is `None`. **Goal:** confirm the model decides if and when to use a tool; we just check the field and skip the dispatch.

In [ ]:
# Sanity check: what happens when the model decides AGAINST using a tool?
messages = [{"role": "user", "content": "Calculate 17 * 23. Answer directly, do NOT use a tool."}]

# 1. Call the LLM
response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
print("=== Response 1 ===")
print(json.dumps(response.message.model_dump(), indent=2, default=str))

# 2. Check: does the model want to call a tool?
print("\n=== tool_calls ===")
print(response.message.tool_calls)

if response.message.tool_calls:
    # 3. Execute the tool
    tc = response.message.tool_calls[0]
    func = tool_map[tc.function.name]
    # ** unpacks the dict into keyword arguments:
    # {"expression": "17 * 23"} becomes calculator(expression="17 * 23")
    result = func(**tc.function.arguments)

    # 4. Send the tool result back to the model
    messages.append(response.message.model_dump())
    messages.append({"role": "tool", "content": result})

    print("\n=== Messages to LLM ===")
    print(json.dumps(messages, indent=2, default=str))

    response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)

print("\n=== Final Response ===")
print(json.dumps(response.model_dump(), indent=2, default=str))

---
### Your Turn

Make each change in the scratchpad below, run it, and briefly note what changed and why.

1. **Remove** the line `messages.append(response.message.model_dump())` from the tool-use loop cell (keep the tool result append). Rerun. What does the final response look like and why?
2. **Change** the prompt in the tool-use loop cell to `"Calculate 17 * 23, then divide the result by 2."` and wrap the tool-call block in `while response.message.tool_calls:` with a safety cap of 5 iterations. How many tool calls does the model make: one combined expression, or two sequential calls?
3. **Change** the prompt in the "answer directly" sanity-check cell to `"Tell me a short joke."` (keep `tools=calculator_tool` unchanged). Does the model touch the calculator? Inspect `content` and `tool_calls`: what does the model do when the available tools are irrelevant to the task, and what does that tell you about how tool selection is decided?


In [ ]:
# Your Turn: Part 4
# Copy the relevant cell above, paste it here, modify, and run.


## Summary

1. `ollama.chat()` takes a `messages` list and returns a `ChatResponse` object.
2. The API is stateless. The model only sees the messages we send in each call.
3. With `tools=`, the model can return tool calls. We check for this with `if response.message.tool_calls`.
4. The tool definition (JSON object with name, description, parameter schema) tells the model what is available. The tool implementation (Python function) is executed by us. They are connected via `tool_map`.
5. The tool-use loop (call LLM → check → execute tool → send result back → call LLM again) is what an agent automates.

→ **Exercise 1**: implement this loop.